In [0]:
Landing_layer_path="/Volumes/pricing_data/deltalake/landing"

In [0]:
current_file_name=dbutils.widgets.get("File_name")
current_file_folder_name=current_file_name.replace('-', '_')

In [0]:
Current_file_date=str(spark.sql(f"""select coalesce(date(max(Process_datetime))+1, date('2023-01-01')) as File_date from pricing_data.process_run_logs.pipeline_process_control where Process_file_name='{current_file_name}' and Process_status='Completed'""").collect()[0]['File_date'])

In [0]:
from datetime import datetime

formated_current_file_date=datetime.strptime(Current_file_date, '%Y-%m-%d').strftime('%m%d%Y')

In [0]:
HTTP_Web_Source_base_url="https://retailpricing.blob.core.windows.net/"
HTTP_Web_Source_File_Name=f"{current_file_name}"
HTTP_Web_Source_End_Url=f"/PW_MW_DR_{formated_current_file_date}.csv"

HTTP_Web_Source_url=HTTP_Web_Source_base_url+HTTP_Web_Source_File_Name+HTTP_Web_Source_End_Url

In [0]:
Landing_layer_full_path=f"{Landing_layer_path}/{current_file_folder_name}/{Current_file_date}"

In [0]:

import requests

response = requests.get(HTTP_Web_Source_url)

if response.status_code != 200:
    raise Exception(f"File not available: {HTTP_Web_Source_url}")

file_content = response.content

dbutils.fs.put(
    Landing_layer_full_path,
    file_content.decode("utf-8"),
    overwrite=True
)

In [0]:
spark.sql(f"""insert into pricing_data.process_run_logs.pipeline_process_control values(
    '{current_file_name}',
    '{Current_file_date}',
    'Completed',
    current_timestamp()
)""")